# v3.5 RT-DETRv4 Checkpoint Inventory and Stub Engine

Documents the RT-DETRv4 checkpoint download status and explains the `_stub` engine situation.

**Context:** RT-DETRv4 (Baidu, 2024) uses a custom training/inference framework not yet integrated.  
**Checkpoint sizes cached:** 169 MB / 318 MB / 505 MB / 1010 MB (S/M/L/X variants)  
**Status:** Checkpoints available; engine implementation blocked on custom CUDA ops.  
**License:** Apache-2.0 (RT-DETRv4, Baidu)

In [ ]:
import pathlib, json, warnings
warnings.filterwarnings('ignore')

artifact = pathlib.Path('notebook/99_final_report/artifacts/v35/rtdetrv4_attempt.json')
if artifact.exists():
    result = json.loads(artifact.read_text())
    print(json.dumps(result, indent=2))
else:
    print('Artifact not found')

In [ ]:
# Checkpoint inventory
import pathlib

cache_dir = pathlib.Path.home() / '.cache/visionservex/rtdetrv4'
EXPECTED = {
    'rtdetrv4_r18vd_120e_coco.pth':  169,
    'rtdetrv4_r34vd_120e_coco.pth':  318,
    'rtdetrv4_r50vd_120e_coco.pth':  505,
    'rtdetrv4_r101vd_60e_coco.pth': 1010,
}

print(f'Cache directory: {cache_dir}')
print(f'{'Checkpoint':<45} {'Expected (MB)':>15} {'Actual (MB)':>12} {'Status':>8}')
print('-' * 85)
for fname, exp_mb in EXPECTED.items():
    fpath = cache_dir / fname
    if fpath.exists():
        actual_mb = fpath.stat().st_size / 1e6
        status = 'OK' if abs(actual_mb - exp_mb) / exp_mb < 0.05 else 'SIZE_MISMATCH'
    else:
        actual_mb = 0
        status = 'MISSING'
    print(f'{fname:<45} {exp_mb:>15} {actual_mb:>12.1f} {status:>8}')

In [ ]:
# Demonstrate _stub engine behavior
from visionservex import VisionModel
from visionservex.exceptions import EngineNotReadyError

try:
    m = VisionModel('rtdetrv4-l')
    print(f'Engine: {m.engine_name}')
    print(f'Status: {m.engine_status}')
except EngineNotReadyError as e:
    print(f'EngineNotReadyError: {e}')
except Exception as e:
    print(f'{type(e).__name__}: {e}')

## Checkpoint Inventory Summary

| Variant | Backbone | Size | COCO mAP | Cache Status |
|---|---|---|---|---|
| rtdetrv4-s | R-18-vd | 169 MB | 46.5 | cached |
| rtdetrv4-m | R-34-vd | 318 MB | 49.2 | cached |
| rtdetrv4-l | R-50-vd | 505 MB | 53.0 | cached |
| rtdetrv4-x | R-101-vd | 1010 MB | 54.3 | cached |

## _stub Engine Situation

The `_stub` engine is a placeholder that:
1. Validates that the checkpoint exists
2. Returns a `EngineNotReadyError` with a descriptive message
3. Logs the blocker reason for tracking

### Blocker: Custom CUDA Ops

RT-DETRv4 uses `paddle_custom_ops` or `torch_custom_ops` for its deformable attention implementation. The next step is to:
1. Port the deformable attention to standard PyTorch ops
2. Or use the ONNX export path (RT-DETR v1 ONNX is already available)

### Next Steps for v3.6

- Implement `rtdetrv4_onnx` engine using exported ONNX graphs
- Fall back to HF `PekingU/rtdetr_r50vd_coco_o365` for v3.6 if custom ops remain blocked